## Importações:  

In [1]:
import pandas as pd
import numpy as np
import sys
import time
import tracemalloc
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, 
                             mean_absolute_error, mean_squared_error, r2_score, median_absolute_error)

sys.path.append('../src')
from expert_system import MotorInferencia


# Função de medição de eficiência

In [2]:
def medir_inferencia(funcao_predicao, X, n_rep=30):
    """Mede tempo e pico de memória apenas da etapa de inferência."""
    tempos = []
    tracemalloc.start()
    for _ in range(n_rep):
        inicio = time.perf_counter()
        funcao_predicao(X)
        tempos.append(time.perf_counter() - inicio)
    
    pico = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    
    media = np.mean(tempos)
    desvio = np.std(tempos)
    
    return {
        "tempo_medio_s": media,
        "tempo_desvio_s": desvio,
        "tempo_por_amostra_ms": 1000 * media / len(X),
        "vazao_amostras_s": len(X) / media,
        "pico_memoria_kb": pico / 1024,
    }

## Protocolo Experimental - CLASSIFICAÇÃO

In [3]:
df_class = pd.read_csv('../data/processed/dataset_classification.csv')
X_class = df_class.drop(columns=['TARGET'])
y_class = df_class['TARGET']

# Divisão 80/20 com os nomes corretos
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

motor_class = MotorInferencia(tipo='classificacao')

# Previsões nos dados de Treino e Teste usando as variáveis corretas
y_pred_train_c = motor_class.prever(X_train_class)
y_pred_test_c = motor_class.prever(X_test_class)

# Cálculo das Métricas
print(f"Acurácia Treino : {accuracy_score(y_train_class, y_pred_train_c):.4f}")
print(f"Acurácia Teste  : {accuracy_score(y_test_class, y_pred_test_c):.4f}")
print(f"F1-score Teste  : {f1_score(y_test_class, y_pred_test_c, zero_division=0):.4f}")
print("Matriz de Confusão (Teste):\n", confusion_matrix(y_test_class, y_pred_test_c))

# Medição Computacional (30 repetições na inferência)
medidas_c = medir_inferencia(motor_class.prever, X_test_class)
print("\n--- Eficiência Computacional ---")
for k, v in medidas_c.items():
    print(f"{k}: {v:.6f}")


Acurácia Treino : 0.2082
Acurácia Teste  : 0.2086
F1-score Teste  : 0.3344
Matriz de Confusão (Teste):
 [[  494 39469]
 [   99  9938]]

--- Eficiência Computacional ---
tempo_medio_s: 4.732186
tempo_desvio_s: 0.083532
tempo_por_amostra_ms: 0.094644
vazao_amostras_s: 10565.941794
pico_memoria_kb: 9112.879883


## Protocolo Experimental - REGRESSÃO

In [4]:
df_reg = pd.read_csv('../data/processed/dataset_regression.csv')
X_reg = df_reg.drop(columns=['TARGET'])
y_reg = df_reg['TARGET']

# Divisão 80/20 idêntica para todos os modelos (sem estratificação)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

motor_reg = MotorInferencia(tipo='regressao')

# Previsões nos dados de Treino e Teste
y_pred_train_r = motor_reg.prever(X_train_r)
y_pred_test_r = motor_reg.prever(X_test_r)

# Cálculo das Métricas
rmse_treino = np.sqrt(mean_squared_error(y_train_r, y_pred_train_r))
rmse_teste = np.sqrt(mean_squared_error(y_test_r, y_pred_test_r))
print(f"RMSE Treino : {rmse_treino:.4f}")
print(f"RMSE Teste  : {rmse_teste:.4f}")
print(f"MAE Teste   : {mean_absolute_error(y_test_r, y_pred_test_r):.4f}")
print(f"R2 Teste    : {r2_score(y_test_r, y_pred_test_r):.4f}")

# Medição Computacional (30 repetições na inferência)
medidas_r = medir_inferencia(motor_reg.prever, X_test_r)
print("\n--- Eficiência Computacional ---")
for k, v in medidas_r.items():
    print(f"{k}: {v:.6f}")

RMSE Treino : 30831.2919
RMSE Teste  : 30713.1240
MAE Teste   : 15573.7968
R2 Teste    : -951154068.9755

--- Eficiência Computacional ---
tempo_medio_s: 6.232040
tempo_desvio_s: 0.043492
tempo_por_amostra_ms: 0.124641
vazao_amostras_s: 8023.055108
pico_memoria_kb: 9447.682617
